# Chapter `2.1` - Model Context Protocol

### Importing the necessary libraries

In [1]:
# Base utils.
import sys
from pathlib import Path
from dotenv import load_dotenv

from IPython.display import Markdown

# Model init. and invocation
from langchain_core.messages import BaseMessage, HumanMessage
from langchain.agents import create_agent

# MCP server
from langchain_mcp_adapters.client import MultiServerMCPClient
import langchain_mcp_adapters.sessions as lc_sessions
import mcp.client.stdio as mcp_stdio

In [2]:
load_dotenv()

# TODO: Environment variable config.

True

### Reqired m/c setup for MCP server

In [3]:
def _patched_stdio_client(server, errlog=None):
    return _orig_stdio_client(server, errlog=MCP_ERRLOG if errlog is None else errlog)


MCP_ERRLOG = open(r"./logs/mcp_stderr.log", "a", encoding="utf-8", buffering=1)
_orig_stdio_client = mcp_stdio.stdio_client

# NOTE: Patch both references used by adapters
mcp_stdio.stdio_client = _patched_stdio_client
lc_sessions.stdio_client = _patched_stdio_client # type: ignore

print("Patch active:", lc_sessions.stdio_client is _patched_stdio_client) # type: ignore

Patch active: True


## MCP Server

### 1. Local

#### Creating MCP client for local MCP server

In [4]:
server_script = Path.cwd() / "resources" / "mcp_server.py"
print(f'MCP server {"exists" if server_script.exists() else "NOT found"} at: {server_script.relative_to(Path.cwd())}')

MCP server exists at: resources\mcp_server.py


In [5]:
client = MultiServerMCPClient(
    {
        "local_server": {
            "transport": "stdio",
            "command": sys.executable,
            "args": [str(server_script)],
            "cwd": str(server_script.parent),
        }
    }
)

tools = await client.get_tools()
resources = await client.get_resources("local_server")
prompt_messages = await client.get_prompt("local_server", "prompt")

print(prompt_messages[0].content if prompt_messages else "No prompt")


    You are a helpful assistant that answers user questions about LangChain, LangGraph and LangSmith.

    You can use the following tools/resources to answer user questions:
    - search_web: Search the web for latest available information.
    - github_file: Access the langchain-ai repo files for obtaining information from the official GitHub repo.

    Rules to be taken into consideration while responding to user queries:
    1. If the user asks a question that is not related to LangChain, LangGraph or LangSmith, you should say "Sorry, I can only answer questions related to LangChain, LangGraph and LangSmith."
    2. You may try multiple tool and resource calls to respond to the user's query.
    3. You may also ask clarifying questions to the user to get a better understanding of their query.
    


#### Configuring the agent

In [6]:
def _msg_to_text(m: BaseMessage) -> str:
    if isinstance(m.content, str):
        return m.content

    return "\n".join(part if isinstance(part, str) else str(part) for part in m.content)

system_prompt_text = "\n\n".join(_msg_to_text(m) for m in prompt_messages).strip()

##### Adding the model

In [7]:
model = "ollama:gemma4:31b-cloud"

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=system_prompt_text
)

##### Prompting

In [8]:
config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Give me whatever information you have about the langchain-mcp-adapters library.")]},
    config=config # type: ignore
)

##### Response

In [9]:
Markdown(response['messages'][-1].content)

The `langchain-mcp-adapters` library is a lightweight wrapper designed to bridge **Anthropic's Model Context Protocol (MCP)** with **LangChain** and **LangGraph**. 

Its primary purpose is to allow developers to use tools hosted on MCP servers as standard LangChain tools, eliminating the need to write custom integration code for every new MCP server.

### Key Capabilities
*   **Tool Conversion:** It converts MCP tools into LangChain-compatible tools that can be plugged directly into LangGraph agents or other LangChain chains.
*   **Multi-Server Connectivity:** It includes a client implementation that enables a single application to connect to multiple MCP servers simultaneously and aggregate their tools.
*   **Prompt Adaptation:** It provides utilities to convert MCP prompts into LangChain messages.
*   **Transport Support:** For the JavaScript/TypeScript version, it supports both `stdio` and `SSE` (Server-Sent Events) transports.

### Installation
Depending on your environment, you can install it via:

*   **Python:**
    ```bash
    pip install langchain-mcp-adapters
    ```
*   **JavaScript/TypeScript:**
    The library is available as `@langchain/mcp-adapters` (now part of the main LangChainJS monorepo).

### Use Case
This library is particularly useful when you want to leverage the growing ecosystem of pre-built MCP servers (which provide standardized ways to access databases, APIs, and local files) without having to manually define the tool schemas and logic within your LangChain project.

### 2. Online

In [14]:
mcp_config = {
    "mcpServers": [
        {
            "pdf-tools-mcp": {
                "transport": "stdio",
                "command": "npx",
                "args": [
                    "-y",
                    "@danielkennedy1/pdf-tools-mcp",
                ],
            },
        },
        {
            "time": {
                "transport": "stdio",
                "command": sys.executable,
                "args": [
                    "-m",
                    "mcp_server_time",
                    "--local-timezone=Asia/Kolkata",
                ],
            }
        },
    ]
}

#### Using the [Time](https://mcp.so/servers/time) MCP server.

In [11]:
time_mcp_client = MultiServerMCPClient(mcp_config["mcpServers"][1])  # type: ignore
tools = await time_mcp_client.get_tools()

agent_online_mcp = create_agent(
    model=model,
    tools=tools,
)

In [12]:
question = HumanMessage(content="What time is it in Noida, Uttar Pradesh? Only provide the time in 12-hour format.")
response = await agent_online_mcp.ainvoke({"messages": [question]})

In [13]:
Markdown(response['messages'][-1].content)

9:34 AM